# Visualization Agent — Agent Approach

## Step 1 — Load the LLM

In [ ]:
import os
import plotly.graph_objects as go
import plotly.io as pio
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

pio.renderers.default = "notebook_connected"

llm = ChatGroq(
    model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
    temperature=0,
)

print(f"✅ LLM loaded: {llm.model_name}")

## Step 2 — Pydantic Schema

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ChartInput(BaseModel):
    """Input schema for the draw_chart tool."""
    chart_type : Literal["bar", "pie", "line"] = Field(description="Best chart type for the data.")
    x_label    : str                           = Field(description="X-axis label, e.g. 'Month'.")
    y_label    : str                           = Field(description="Y-axis label, e.g. 'Sales'.")
    title      : str                           = Field(description="Short descriptive chart title.")
    x_values   : list[str]                     = Field(description="X-axis category labels.")
    y_values   : list[float]                   = Field(description="Numeric values for each category.")

print("✅ ChartInput schema defined")

## Step 3 — @tool decorated function

In [ ]:
from langchain_core.tools import tool

@tool(args_schema=ChartInput)
def draw_chart(chart_type: str, x_label: str, y_label: str,
               title: str, x_values: list, y_values: list) -> str:
    """Renders a visualization chart from the given data."""

    if chart_type == "bar":
        fig = go.Figure(go.Bar(x=x_values, y=y_values, marker_color="steelblue"))
        fig.update_layout(xaxis_title=x_label, yaxis_title=y_label)

    elif chart_type == "line":
        fig = go.Figure(go.Scatter(
            x=x_values, y=y_values,
            mode="lines+markers",
            line=dict(color="steelblue", width=3),
            marker=dict(size=8)
        ))
        fig.update_layout(xaxis_title=x_label, yaxis_title=y_label)

    elif chart_type == "pie":
        fig = go.Figure(go.Pie(labels=x_values, values=y_values, hole=0.3))

    fig.update_layout(title=title, template="plotly_dark", height=450)
    fig.show()

    return f"Chart rendered: [{chart_type}] — '{title}'"

print("✅ draw_chart tool defined")

## Step 4 — Build the Agent

System prompt explicitly tells the LLM:
- Pick ONE chart type only
- Call the tool ONCE then stop

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a data visualization assistant.

When the user provides data:
1. Choose EXACTLY ONE chart type using these rules:
   - If the user specifies a chart type (bar/line/pie), use that.
   - If no chart type is mentioned:
       * Use 'line' for time-series or trend data (e.g. monthly, yearly).
       * Use 'bar' for category comparisons.
       * Use 'pie' for part-of-whole proportions.

2. Call the draw_chart tool EXACTLY ONCE with the chosen type.
3. After the tool returns, stop immediately. Do not call the tool again."""

agent = create_react_agent(
    model  = llm,
    tools  = [draw_chart],
    prompt = SYSTEM_PROMPT,
)

print("✅ Agent built")

## Step 5 — Run

`recursion_limit=5` prevents infinite looping.
The agent needs at most 3 steps: think → call tool → stop.

In [ ]:
USER_INPUT = (
    "Show me these values in a visualization graph: monthly sales and product. "
    "Monthly sales of product X: May 200, June 500, July 100, August 55."
)

result = agent.invoke(
    {"messages": [("human", USER_INPUT)]},
    config={"recursion_limit": 5},   # stops infinite loops
)

print("\n" + "─" * 50)
print(f"Agent output: {result['messages'][-1].content}")